# 1. Prepare data

In [1]:
import pandas as pd
import numpy as np
# import the matplotlib.pyplot library and name it as plt
import matplotlib.pyplot as plt
# #import the PdfPages to save all plots
import os
from matplotlib.backends.backend_pdf import PdfPages
import math
from scipy import stats

### 1.1 Extract data


In [3]:
def read_clinical():
    # Define the columns you want to keep
    columns_to_keep = ['sub_id', 'age', 'residenc', 'redcap_event_name', 'cdrglob','date_screen', 'othcondx', 'cogothx']

    file =pd.read_excel("clin_fin_red.xlsx",usecols=columns_to_keep)
    return file

In [4]:
clinical=read_clinical()
len(clinical['sub_id'].unique())

236

We have 236 users from clinical dataset 

In [7]:
### mapping home_id to subject_id
def read_mapping(study):
    # Get a list of all files in the current directory
    files_in_directory = os.listdir('.')
    patient_file = pd.read_csv(f"{study}-CART_Home-Subject_Dates_Through_2024-03-03.csv")
    
    return patient_file

In [8]:
OHSU_mapping=read_mapping('OHSU')
RUSH_mapping=read_mapping('RUSH')
VA_mapping=read_mapping('VA')
MIAMI_mapping=read_mapping('MIAMI')

In [9]:
# Concatenate all mapping DataFrames
full_mapping_df = pd.concat([OHSU_mapping, RUSH_mapping, VA_mapping, MIAMI_mapping])

# home_id from sensor data

In [10]:
# # Filter the DataFrame based on the list of home_ids
# filtered_data = OHSU_mapping[OHSU_mapping['home_id'].isin(OHSU)]
# filtered_data

In [11]:
def find_duplicate(study): # find homes that have more than one subject
    data_mapping=read_mapping(study)
    grouped = data_mapping.groupby('home_id')
    duplicate_home_ids = grouped.filter(lambda x: x['sub_id'].nunique() > 1)
    return duplicate_home_ids

In [12]:
# Create a mapping from home_id to sub_id
def replace_sub(study):
    data_mapping=read_mapping(study)
    id_mapping = dict(zip(data_mapping['home_id'], data_mapping['sub_id']))
    # Replace home_id in the list with sub_id using the mapping
    if study=='OHSU':
        data = OHSU_home_id
    if study=='VA':
        data = VA_home_id
    if study=='RUSH':
        data = RUSH_home_id
    if study=='MIAMI':
        data = MIAMI_home_id
        
    abc_sub_ids = [id_mapping.get(home_id, None) for home_id in data]
    return abc_sub_ids

In [13]:
OHSU_home_id=[1093, 1094, 1095, 1096, 1097, 1102, 1105, 1106, 1135, 1430, 
      1450, 1451, 1453, 1468, 1473, 1474, 1475, 1477, 1478, 1481, 
      1482, 1484, 1487, 1488, 1494, 1496, 1497, 1498, 1504, 1505, 
      1506, 1507, 1508, 1509, 1510, 1511, 1514, 1516, 1523, 1524, 
      1525, 1526, 1527, 1537, 1538, 1539, 1540, 1543, 1554, 1555, 
      1562, 1564, 1590, 1595, 1598, 1599, 1615, 1616, 1618, 1619, 
      1622, 1653, 1671, 1693, 1694, 1695, 1717, 1719, 1843, 1847, 
      1879, 1880, 1890, 1905, 1922, 1934, 1954, 1964, 2014, 922, 931]
len(OHSU_home_id)

81

In [14]:
# remove duplicate---homes that have more than one subject
duplicate_sub=find_duplicate(study='OHSU')
OHSU_home_id=[a for a in OHSU_home_id if a not in duplicate_sub['home_id'].unique() ]
# replace home_id to subject_id
OHSU_sub=replace_sub('OHSU')
len(OHSU_home_id)

(76, 76)

In [16]:
VA_home_id= [1100, 1454, 1456, 1457, 1458, 1462, 1463, 1464, 1466, 1471, 
     1472, 1479, 1480, 1486, 1489, 1495, 1499, 1500, 1503, 1520, 
     1521, 1522, 1550, 1551, 1566, 1569, 1570, 1576, 1579, 1583, 
     1601, 1603, 1604, 1606, 1607, 1608, 1609, 1610, 1625, 1626, 
     1627, 1628, 1629, 1630, 1631, 1632, 1655, 1657, 1658, 1659, 
     1660, 1661, 1662, 1663, 1664, 1665, 1672, 1674, 1697, 1698, 
     1713, 1724, 1743, 1788, 1790, 1846, 1876, 1885, 1913, 1952, 
     1957, 1971, 2010, 2021, 2022, 2034]
len(VA_home_id)

76

In [17]:
# remove duplicate---homes that have more than one subject
duplicate_sub=find_duplicate(study='VA')
VA_home_id=[a for a in VA_home_id if a not in duplicate_sub['home_id'].unique() ]
# replace home_id to subject_id
VA_sub=replace_sub('VA')
len(VA_home_id),len(VA_sub)

(21, 21)

In [18]:
RUSH_home_id=[1517, 1518, 1519, 1529, 1530, 1531, 1547, 1548, 1549, 1561, 
       1563, 1567, 1568, 1573, 1575, 1577, 1578, 1589, 1597, 1640, 
       1690, 1691, 1710, 1711, 1712, 1714, 1715, 1728, 1729, 1730, 
       1749, 1750, 1757, 1759, 1760, 1762, 1763, 1764, 1766, 1769, 
       1774, 1782, 1787, 1789, 1798, 1801, 1804, 1811, 1813, 1816, 
       1821, 1823, 1835, 1844, 1853, 1861, 1872, 1916, 1949, 1960, 
       1968, 1969]
len(RUSH_home_id)

62

In [19]:
# remove duplicate---homes that have more than one subject
duplicate_sub=find_duplicate(study='RUSH')
RUSH_home_id=[a for a in RUSH_home_id if a not in duplicate_sub['home_id'].unique() ]
# replace home_id to subject_id
RUSH_sub=replace_sub('RUSH')
len(RUSH_home_id),len(RUSH_sub)

(53, 53)

In [20]:
MIAMI_home_id=[1541, 1542, 1544, 1559, 1560, 1571, 1572, 1582, 1586, 1591,
       1592, 1600, 1612, 1617, 1699, 1704, 1706, 1707, 1751, 1754, 
       1778, 1815, 1838, 1883, 1897, 1962, 1966, 1985]
len(MIAMI_home_id)

28

In [21]:
# remove duplicate---homes that have more than one subject
duplicate_sub=find_duplicate(study='MIAMI')
MIAMI_home_id=[a for a in MIAMI_home_id if a not in duplicate_sub['home_id'].unique() ]
# replace home_id to subject_id
MIAMI_sub=replace_sub('MIAMI')
len(MIAMI_home_id),len(MIAMI_sub)

(24, 24)

In [22]:
patients =OHSU_sub+VA_sub+RUSH_sub+MIAMI_sub
len(patients)

174

###  Find 'sub_id' with more than one 'home_id'

In [23]:
# Find 'sub_id' with more than one 'home_id'
duplicate_sub_ids = full_mapping_df.groupby('sub_id').filter(lambda x: x['home_id'].nunique() > 1)['sub_id'].unique()
duplicate_sub_ids

array([1191, 1666, 1674, 1695, 1701, 1705, 2049, 1712, 1847, 1848, 2102,
       1357, 1717, 1718, 1855, 1897, 1898, 1895, 1896, 1899, 1909, 1910,
       2031])

### Remove stange subjects (have multiple homes)

In [24]:
# remove stange subjects (have multiple homes)
patients=[a for a in patients if a not in duplicate_sub_ids]
len(patients)

154

In [25]:
clinical = clinical[clinical['sub_id'].isin(patients)]
clinical=clinical.sort_values(by=['sub_id', 'date_screen']).reset_index(drop=True)
# clinical

## Filtering clinical data

In [26]:
# Step1: find those we have both clinical and sensor data
clinical = clinical[clinical['sub_id'].isin(patients)]
clinical=clinical.sort_values(by=['sub_id', 'date_screen']).reset_index(drop=True)
# clinical

len(clinical['sub_id'].unique())
#save to a csv file
# clinical.to_csv('filtered_clinical.csv')

102

In [27]:
# # Step1: Filter the DataFrame with those start as healthy
# # Filter for 'baseline_visit_arm_1' in 'redcap_event_name' AND 'cogothx' is NaN
# criteria_sub_ids = clinical[(clinical['redcap_event_name'] == 'baseline_visit_arm_1') & pd.isna(clinical['cogothx'])]['sub_id'].unique()
# clinical = clinical[clinical['sub_id'].isin(criteria_sub_ids)]
 
# clinical.reset_index(drop=True)
# len(clinical['sub_id'].unique())

# Step2: Filter the DataFrame with those residenc=1
criteria_sub_ids = clinical[(clinical['redcap_event_name'] == 'baseline_visit_arm_1')][clinical['residenc']==1.0]['sub_id'].unique()

# criteria_sub_ids = clinical[clinical['residenc']==1.0]['sub_id'].unique()
clinical = clinical[clinical['sub_id'].isin(criteria_sub_ids)]
clinical.reset_index(drop=True)
# len(clinical['sub_id'].unique())


criteria_sub_ids = clinical[clinical['residenc']==1.0]['sub_id'].unique()
clinical = clinical[clinical['sub_id'].isin(criteria_sub_ids)]
clinical.reset_index(drop=True)
len(clinical['sub_id'].unique())

<ipython-input-27-03f7afdbdfca>:10: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  criteria_sub_ids = clinical[(clinical['redcap_event_name'] == 'baseline_visit_arm_1')][clinical['residenc']==1.0]['sub_id'].unique()


49

In [28]:
# clinical[clinical['residenc']!=1.0]

In [29]:
# Filter the DataFrame to keep only sub_ids with at least 2 rows
clinical = clinical.groupby('sub_id').filter(lambda x: len(x) >= 2)

# Reset index after filtering
clinical.reset_index(drop=True, inplace=True)

clinical_patient = clinical['sub_id'].unique()
len(clinical_patient)

37

In [30]:
# Step3: Find MCI_patients (at leaset one 'cogothx' not NA)
MCI_patients = clinical.dropna(subset=['cogothx'])['sub_id'].unique()
len(MCI_patients)

9

In [31]:
# Step4: Find Healthy users
healthy = clinical[~clinical['sub_id'].isin(MCI_patients)]['sub_id'].unique()
len(healthy)

28

### Add home_id to clinical

In [34]:
# Merging the clinical DataFrame with OHSU_mapping to add home_id
clinical = clinical.merge(full_mapping_df[['sub_id', 'home_id']], on='sub_id', how='left')
# clinical_with_home_id

In [35]:
# Merging the clinical DataFrame with OHSU_mapping to add home_id
clinical = clinical.merge(full_mapping_df[['home_id', 'study']], on='home_id', how='left')
# clinical

In [36]:
clinical = clinical[clinical['sub_id'].isin(clinical_patient)]
clinical=clinical.sort_values(by=['sub_id', 'date_screen']).reset_index(drop=True)

In [37]:
len(clinical_patient)

37

In [38]:
clinical[clinical['redcap_event_name']=='baseline_visit_arm_1']

,sub_id,redcap_event_name,date_screen,age,residenc,othcondx,cdrglob,cogothx,home_id,study
0,1113,baseline_visit_arm_1,2018-06-13,87.8,1.0,NaN,0.0,NaN,1135,OHSU-CART
2,1623,baseline_visit_arm_1,2018-02-27,74.4,1.0,NaN,0.0,NaN,1450,OHSU-CART
5,1655,baseline_visit_arm_1,2018-04-02,70.7,1.0,NaN,0.5,MCI,1464,VA-CART
8,1702,baseline_visit_arm_1,2018-06-08,75.7,1.0,NaN,0.0,NaN,1497,OHSU-CART
10,1703,baseline_visit_arm_1,2018-06-11,70.2,1.0,NaN,0.0,NaN,1498,OHSU-CART
12,1704,baseline_visit_arm_1,2018-06-13,71.4,1.0,Cataracts,0.0,NaN,1504,OHSU-CART
14,1711,baseline_visit_arm_1,2018-06-22,73.2,1.0,Cataracts,0.0,NaN,1507,OHSU-CART
16,1714,baseline_visit_arm_1,2018-06-27,67.1,1.0,NaN,0.0,NaN,1511,OHSU-CART
18,1725,baseline_visit_arm_1,2018-07-06,62.4,1.0,"Hepatitis C, TBI,",0.0,NaN,1514,OHSU-CART
20,1726,baseline_visit_arm_1,2018-07-05,62.1,1.0,NaN,0.0,NaN,1522,VA-CART


In [41]:
# # Find rows where 'cdrglob' is NaN
# no_data=list(clinical[clinical['cdrglob'].isna()]['sub_id'].unique())
# # no_data

In [42]:
# clinical_patient=[p for p in clinical_patient if p not in no_data]

In [39]:
clinical = clinical[clinical['sub_id'].isin(clinical_patient)]
clinical=clinical.sort_values(by=['sub_id', 'date_screen']).reset_index(drop=True)

In [40]:
len(clinical['home_id'].unique())

37

In [41]:
#save to a csv file
clinical.to_csv('filtered_clinical.csv')

### Sensor data

In [43]:
# Create a mapping from home_id to sub_id
def replace_sub_back(study):
    data_mapping=read_mapping(study)
    id_mapping = dict(zip(data_mapping['sub_id'],data_mapping['home_id']))
    # Replace home_id in the list with sub_id using the mapping
    if study=='OHSU':
        data = updated_OHSU
    if study=='VA':
        data = updated_VA
    if study=='RUSH':
        data = updated_RUSH
    if study=='MIAMI':
        data = updated_MIAMI
        
    abc_sub_ids = [id_mapping.get(sub_id, None) for sub_id in data]
    return abc_sub_ids

In [44]:
#subject_id
updated_OHSU=[a for a in clinical_patient if a in OHSU_sub]
# updated_OHSU

In [45]:
#subject_id
updated_VA=[a for a in clinical_patient if a in VA_sub]
# updated_VA

In [46]:
#subject_id
updated_RUSH=[a for a in clinical_patient if a in RUSH_sub]
# updated_RUSH

In [47]:
#subject_id
updated_MIAMI=[a for a in clinical_patient if a in MIAMI_sub]
# updated_MIAMI

In [48]:
# patients_subject
patients = updated_OHSU+updated_VA+updated_RUSH+updated_MIAMI
len(patients)

37

In [49]:
OHSU_home_final=replace_sub_back('OHSU')
OHSU_home_final

[1135, 1450, 1497, 1498, 1504, 1507, 1511, 1514, 1540, 1615, 1693]

In [50]:
VA_home_final=replace_sub_back('VA')
VA_home_final

[1464, 1522, 1603, 1609, 1610, 1607, 1627, 1628, 1657, 1660, 1724]

In [51]:
MIAMI_home_final=replace_sub_back('MIAMI')
MIAMI_home_final

[1542,
 1541,
 1559,
 1571,
 1572,
 1582,
 1586,
 1591,
 1592,
 1600,
 1617,
 1699,
 1706,
 1707,
 1754]

In [52]:
patiens_home=OHSU_home_final+VA_home_final+MIAMI_home_final
patiens_home

[1135,
 1450,
 1497,
 1498,
 1504,
 1507,
 1511,
 1514,
 1540,
 1615,
 1693,
 1464,
 1522,
 1603,
 1609,
 1610,
 1607,
 1627,
 1628,
 1657,
 1660,
 1724,
 1542,
 1541,
 1559,
 1571,
 1572,
 1582,
 1586,
 1591,
 1592,
 1600,
 1617,
 1699,
 1706,
 1707,
 1754]

In [53]:
#home_id
MCI_patients = clinical.dropna(subset=['cogothx'])['home_id'].unique()
updated_MCI_patients=[a for a in MCI_patients if a in patiens_home]
len(updated_MCI_patients),len(MCI_patients)

(9, 9)

In [54]:
updated_MCI_patients

[1464, 1511, 1603, 1609, 1607, 1627, 1628, 1657, 1724]

In [55]:
# criteria_sub_ids=new_patient
# criteria_sub_ids = clinical[clinical['residenc']==1.0]['sub_id'].unique()
# clinical = clinical[clinical['sub_id'].isin(criteria_sub_ids)]
# clinical.reset_index(drop=True)
# clinical

In [56]:
def readData(patient):
    # Get a list of all files in the current directory
    files_in_directory = os.listdir('.')
    matching_files = [file for file in files_in_directory if file.endswith(f"-CART_{patient}.csv")]
    
    # Initialize an empty list to store data from all matching files
    all_data = []

    # Check if there are any matching files
    if not matching_files:
        print(f"No files found for patient {patient}")
        return all_data  # Return an empty list if no files match
    
    # Loop through each matching file
    for file in matching_files:
        try:
            # Read the CSV file
            patient_file = pd.read_csv(file)
#             all_data.extend(tm.values.tolist())
            
        except Exception as e:
            print(f"An error occurred while reading {file}: {e}")
            
    study_value = patient_file['study'].unique()[0]
    return patient_file,study_value


In [57]:
def readarea_id():
    tm = pd.read_excel('area_dictionary.xlsx')
#     area_ids=tm.values.tolist()
    return tm

In [58]:
area_ids=readarea_id()

In [55]:
def read_adjacent(patient): #home_id
    patient_file,study_value=readData(patient)
    #home_id
    VA_adjacent = pd.read_csv('NYCE_Adjacency_VA-CART_2024-03-03.csv')
    RUSH_adjacent = pd.read_csv('NYCE_Adjacency_RUSH-CART_2024-03-03.csv')
    MIAMI_adjacent = pd.read_csv('NYCE_Adjacency_MIAMI-CART_2024-03-03.csv')
    OHSU_adjacent = pd.read_csv('NYCE_Adjacency_OHSU-CART_2024-03-03.csv')
    # Dictionary to map studies to their respective adjacency DataFrame
    study_to_adjacent_df = {
        'VA-CART': VA_adjacent,
        'RUSH-CART': RUSH_adjacent,
        'MIAMI-CART': MIAMI_adjacent,
        'OHSU-CART': OHSU_adjacent}
    
    # Iterate through the unique studies in patient_file and merge with the respective adjacency DataFrame
    for study, adjacent_df in study_to_adjacent_df.items():
   
        # Filter patient_file rows for the current study
        filtered_patient_file = patient_file[patient_file['study'] == study]
        
        # Perform the merge if there are rows for this study
        if not filtered_patient_file.empty:
            filtered_df = adjacent_df[adjacent_df['home_id'] == patient]
            print(patient,'study',study)

    return filtered_df

In [56]:
def clean_info(patient_file):
#     patient_file=readData(1657)

    ### Merge DataFrames on 'areaid'
    merged_df = pd.merge(patient_file, area_ids, on='areaid', how='left')
    merged_df=merged_df.dropna()
    info=merged_df.values.tolist()
    
    # extract time, area_name from and stored all to a list of lists
    out=[]
    for item in info:
        out.append([item[2],item[-1]])
        
        
    from datetime import datetime
    for item in out:
        try:
           # formatting the date using strptime() function
            item[0]==datetime.strptime(item[0],'%Y-%m-%d %H:%M:%S.%f')
            item[0]=(datetime.strptime(item[0],'%Y-%m-%d %H:%M:%S.%f').strftime("%d/%m/%Y %H:%M:%S"))

        # If the date validation goes wrong
        except:
            item[0]=(datetime.strptime(item[0],'%Y-%m-%d %H:%M:%S').strftime("%d/%m/%Y %H:%M:%S"))
            
            
    return out

In [57]:
def convert_timezone(out,value):
#     patient_file,value=readData(patient)
    import pytz
    # Create a DataFrame
    df = pd.DataFrame(out, columns=['timestamp', 'location'])

    # Convert the 'timestamp' column to datetime, assuming the original timezone is UTC
    df['timestamp'] = pd.to_datetime(df['timestamp'], format='%d/%m/%Y %H:%M:%S').dt.tz_localize('UTC')

    
    #OHSU/VA='Boise', RUSH='Chicago' MIAMI='New_York'
    if ('OHSU' in value) or ('VA' in value) :
        timezone='America/Boise'
        print(patient,value,timezone)
    elif ('RUSH' in value):
        timezone='America/Chicago'
        print(patient,value,timezone)
    elif ('MIAMI' in value):
        timezone='America/New_York'
        print(patient,value,timezone)
        
    # Convert the timezone to US/Pacific
    df['timestamp'] = df['timestamp'].dt.tz_convert(timezone)#New_York/Los_Angeles
    # Convert timestamps back to string without timezone for final output
    df['timestamp'] = df['timestamp'].dt.strftime('%d/%m/%Y %H:%M:%S')

    # Convert the updated DataFrame back to the original training_data format
    out = df.values.tolist()
    return out

In [85]:
for patient in patiens_home: #
    patient_file,value=readData(patient)
    out=clean_info(patient_file)
    out=convert_timezone(out,value)
#     print(patient,value)
    #Convert list of lists to list of tuples to make them hashable
    out_tuples = [tuple(entry) for entry in out]
    # Use set to remove duplicates, then convert it back to a list of lists
    unique_out = [list(entry) for entry in set(out_tuples)]
    # sorted the result by the first element (timestamp)
    unique_out_sorted = sorted(unique_out, key=lambda x: x[0])

    df =  pd.DataFrame (out)
    df.to_csv(f"{patient}_cleaned_alldata"'.csv', index=False, header=True)
    
    adjacent_list=read_adjacent(patient).iloc[:, -2:].drop_duplicates()

    adjacent_list.to_csv(f"{patient}_adjacent_list"'.csv', index=False, header=True)
    

1135 OHSU-CART America/Boise
1135 study OHSU-CART
1450 OHSU-CART America/Boise
1450 study OHSU-CART
1497 OHSU-CART America/Boise
1497 study OHSU-CART
1498 OHSU-CART America/Boise
1498 study OHSU-CART
1504 OHSU-CART America/Boise
1504 study OHSU-CART
1506 OHSU-CART America/Boise
1506 study OHSU-CART
1508 OHSU-CART America/Boise
1508 study OHSU-CART
1507 OHSU-CART America/Boise
1507 study OHSU-CART
1511 OHSU-CART America/Boise
1511 study OHSU-CART
1514 OHSU-CART America/Boise
1514 study OHSU-CART
1526 OHSU-CART America/Boise
1526 study OHSU-CART
1540 OHSU-CART America/Boise
1540 study OHSU-CART
1590 OHSU-CART America/Boise
1590 study OHSU-CART
1615 OHSU-CART America/Boise
1615 study OHSU-CART
1618 OHSU-CART America/Boise
1618 study OHSU-CART
1693 OHSU-CART America/Boise
1693 study OHSU-CART
1464 VA-CART America/Boise
1464 study VA-CART
1522 VA-CART America/Boise
1522 study VA-CART
1603 VA-CART America/Boise
1603 study VA-CART
1609 VA-CART America/Boise
1609 study VA-CART
1610 VA-CART Ame